[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/notebooks/04_Visualizing_Data_in_Python/05_viz.ipynb)

# Multi-omics Data Science

# Plotly

## Python Open Source Graphing Library

Plotly is a Python library for building high-quality interactive figures. Here we show how to build the plots you need to explore a data set held in a pandas DataFrame.

![gallery](https://miro.medium.com/max/1458/1*qKpV3vkPZYoffsvFSEuw8A.png)

One advantage of Plotly over other Python plotting libraries is how simply figures can be created and modified, and that they are interactive out of the box. Plotly has an interface called [Plotly Express](https://plotly.com/python/plotly-express/) that simplifies plot creation even further.

Interactivity is not a gimmick in omics. A scatter plot of 1458 protein groups is an anonymous cloud of dots until you can hover over one point and read the gene name and the p-value. Every plot in this notebook is built on the **course data set**, so the figures you make here are the figures you will need for the rest of the week.

# Getting Started

## Import the libraries

**Pandas**, **NumPy** and **Plotly Express**.

We also fix two things once, at the top, and reuse them in every figure: the **order of the three patient groups** and the **colour of each group**. Keeping one palette across a whole figure set is not decoration, it is readability: the reader learns "orange = susceptible infection" once and then reads every later plot for free.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Where the course data lives (the raw files of the course GitHub repository)
COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"

# One group order and one palette for the whole notebook
GROUP_ORDER = ["Con", "CSKP", "CRKP"]
GROUP_COLOURS = {"Con": "#4C72B0", "CSKP": "#DD8452", "CRKP": "#C44E52"}

# Introduction

Serum was collected on the day of admission from **45 septic patients**, 15 in each of three groups:

| Group | Sample IDs | Meaning |
| --- | --- | --- |
| `Con` | `Con1` ... `Con15` | sepsis with negative cultures |
| `CSKP` | `KP1` ... `KP15` | carbapenem-**susceptible** *Klebsiella pneumoniae* sepsis |
| `CRKP` | `CRKP1` ... `CRKP15` | carbapenem-**resistant** *K. pneumoniae* sepsis |

Two omics layers were measured on the same samples: **proteomics** (diaPASEF acquisition processed with DIA-NN, 1458 protein groups) and **metabolomics** (targeted MRM, 1073 named metabolites).

The clinical question is whether host serum molecules can distinguish a **resistant** from a **susceptible** infection on day 0, before the culture result comes back.

We start with the clinical table: one row per patient, with the group, sex, age, BMI, a panel of clinical laboratory values and a set of 0/1 comorbidity flags. Note the `sep="\t"` argument: these are tab-separated (`.tsv`) files, not comma-separated ones.

In [ ]:
meta = pd.read_csv(f"{BASE_URL}/metadata/sample_metadata.tsv", sep="\t")

print("patients:", meta.shape[0], " columns:", meta.shape[1])
meta.head()

In [ ]:
meta.tail()

## The differential abundance table

The second table is the list of proteins the original publication reported as **differentially abundant** between groups. It has one row per protein and per comparison, with the log2 fold change and the p-value.

There are four comparisons: the three pairwise ones (`KP_vs_Con`, `CRKP_vs_Con`, `CRKP_vs_KP`) and a three-group ANOVA (`CRKP_vs_KP_vs_Con`). The ANOVA rows have no fold change, because "a fold change" is not defined for three groups at once, so those cells are empty (`NaN`).

Keep one thing in mind while we plot this table: it contains **only the proteins that passed the significance cut-off** in the publication (every p-value is below 0.05). It is a table of hits, not the full result of the test. We will come back to what that does to a volcano plot.

In [ ]:
deps = pd.read_csv(f"{BASE_URL}/proteomics/data/published_deps.tsv", sep="\t")

print(deps.shape)
print(deps["comparison"].value_counts())
deps.head()

### Two columns we will reuse

Two derived columns make the next plots much easier to read:

- `neg_log10_p` = $-\log_{10}(p)$. Small p-values become large numbers, so "more significant" points move **up** the y axis instead of squeezing into a sliver near zero. This is the standard transformation for any figure involving p-values.
- `direction` says whether the protein is higher or lower in the cases than in the controls of that comparison.

We also keep the pairwise comparisons in their own DataFrame, `deps_pairwise`, and pick the 20 strongest changes of the resistant-versus-culture-negative comparison as a small, plottable subset.

In [ ]:
deps["neg_log10_p"] = -np.log10(deps["p_value"])

# Only the pairwise comparisons have a fold change
deps_pairwise = deps.dropna(subset=["log2_fold_change"]).copy()
deps_pairwise["direction"] = np.where(
    deps_pairwise["log2_fold_change"] > 0, "up in cases", "down in cases"
)

# The 20 strongest changes between resistant infection and culture-negative sepsis
crkp_vs_con = deps_pairwise[deps_pairwise["comparison"] == "CRKP_vs_Con"]
top20_crkp_con = (
    crkp_vs_con
    .reindex(crkp_vs_con["log2_fold_change"].abs().sort_values(ascending=False).index)
    .head(20)
    .sort_values("log2_fold_change")
)

# The 10 strongest changes of each pairwise comparison
top_per_comparison = (
    deps_pairwise
    .assign(abs_log2fc=lambda d: d["log2_fold_change"].abs())
    .sort_values("abs_log2fc", ascending=False)
    .groupby("comparison")
    .head(10)
    .sort_values(["comparison", "log2_fold_change"])
)

top20_crkp_con[["genes", "log2_fold_change", "p_value", "direction"]]

## The protein matrix

The third table is the quantitative matrix: one row per protein group, one column per sample, and the measured intensity in each cell.

Two details of this table matter for every plot we will make from it:

1. The last three columns (`QC_pool1` ... `QC_pool3`) are **pooled quality-control injections**, not patients. They are the same sample measured three times to check instrument stability, so they must be kept out of any biological comparison. We separate them from the 45 patient columns straight away.
2. About 27 % of the cells are empty. In mass spectrometry a missing value usually means "below the detection limit in that run", not "measured as zero". Plotly simply skips missing values, but you should always know how many there are.

In [ ]:
prot = pd.read_csv(f"{BASE_URL}/proteomics/data/protein_groups_matrix.tsv", sep="\t")

info_cols = ["protein_group", "protein_names", "genes", "description"]
qc_cols = [col for col in prot.columns if col.startswith("QC")]
patient_cols = [col for col in prot.columns if col not in info_cols + qc_cols]

print("protein groups:", prot.shape[0])
print("patient samples:", len(patient_cols), " QC pools:", len(qc_cols))
print("missing values: {:.1%}".format(prot[patient_cols].isna().to_numpy().mean()))
prot[info_cols + patient_cols[:4]].head()

# Line charts

We can pass the values we want to plot as two lists (here, two columns of the DataFrame).

In [ ]:
px.line(x = top20_crkp_con["genes"], y = top20_crkp_con["log2_fold_change"])

We can also use the pandas DataFrame directly and name the columns.

In [ ]:
px.line(data_frame = top20_crkp_con, x = "genes" , y = "log2_fold_change")

## Store it in a variable

If you want to reuse the plot, show it several times or modify it later, you can store it in a variable like any other value or data structure. To display the plot, simply use `.show()`.

**A word on the plot we just made.** A line implies that the points in between mean something: it says "as x increases, y follows". Here x is a list of gene names, which have no natural order at all, so the line is a purely visual device that guides the eye along a ranking. Plots like this are common (they are sometimes called waterfall plots) and they are fine as long as you sort the genes deliberately, but for categorical x a **bar chart** or a **dot plot** is usually the more honest choice. We will build both below.

In [ ]:
fig = px.line(data_frame = top20_crkp_con, x = "genes" , y = "log2_fold_change")

In [ ]:
fig.show()

## Colour

You can use another column of the DataFrame to colour your plot. Here we take the 10 strongest changes of each of the three pairwise comparisons and give each comparison its own line.

In [ ]:
fig = px.line(data_frame = top_per_comparison, x = "genes" , y = "log2_fold_change", color="comparison")

In [ ]:
fig.show()

## Structure of a Plotly figure

In Plotly a figure is really a dictionary, which you can inspect with `.data` on the variable holding the plot, and with `.layout` to see the layout of the figure.

In [ ]:
#fig.data

In [ ]:
fig.layout

As you can see there are many parameters in the data and layout dictionaries that can be modified. Usually, to change the appearance of a figure, we modify the layout dictionary. We can do that with `.update_layout()`, for example to add a title or rename an axis.

Do this for every figure you intend to show to somebody else. `log2_fold_change` is a column name; **"log2 fold change (CRKP / Con)"** is an axis label. An axis without a name and a unit is the single most common defect in omics figures: only the person who wrote the code can read it.

In [ ]:
fig.update_layout(template="plotly_dark",
                  title = "Strongest differentially abundant serum proteins",
                  xaxis_title="Gene",
                  yaxis_title="log2 fold change (case / control)")

## Text argument

We can add the value of a variable at the coordinates given by the x and y argument by using the `text` argument.

In [ ]:
fig = px.line(top20_crkp_con,
              x="genes",
              y="log2_fold_change",
              text="genes", # The text argument writes the label on the data point itself
              title="CRKP vs Con: 20 strongest fold changes")
fig.show()

In [ ]:
fig = px.line(top_per_comparison,
              x="genes",
              y="log2_fold_change",
              color="direction",
              title="Proteins up and down in the infected groups")
fig.show()

# Exercises

1) Create a line chart showing the significance (`neg_log10_p`) per gene, coloured by comparison. Use `top_per_comparison` so the figure stays readable.

2) Make the same plot but only for the proteins of the comparison `CRKP_vs_KP` (resistant versus susceptible infection) with a p-value below 0.01, and without using colour.

3) Change the layout so it uses the Plotly dark template (see above for a hint).

# Scatter plots

Scatter plots use the x and y coordinates to show the relationship between two variables. They are the workhorse of omics: a volcano plot, an MA plot, a PCA score plot and a QC plot of mean against variability are all scatter plots with well-chosen axes.

In Plotly the function `px.scatter()` draws scatter plots and takes parameters similar to those of the line plot.

The first example puts the mean abundance in the cases on one axis and the mean abundance in the controls on the other. Both are already on a log2 scale. Points far from the diagonal are the proteins that change; the colour repeats that information as the fold change.

In [ ]:
fig = px.scatter(deps_pairwise,
                 x="mean_control",
                 y="mean_case",
                 color="log2_fold_change",
                 color_continuous_scale="RdBu_r",
                 color_continuous_midpoint=0,
                 hover_name="genes",
                 labels={"mean_control": "mean log2 intensity, controls",
                         "mean_case": "mean log2 intensity, cases",
                         "log2_fold_change": "log2 FC"},
                 title="Case vs control abundance of the differential proteins")
fig.show()

## The symbol parameter

If we want to use different symbols in the scatter plot to distinguish between categories, we can pass the `symbol` argument and give it the name of a column.

To keep the figure simple we first filter the table down to two comparisons.

In [ ]:
deps2comp = deps_pairwise[deps_pairwise["comparison"].isin(["CRKP_vs_Con", "CRKP_vs_KP"])]
fig = px.scatter(deps2comp,
                 x="mean_control",
                 y="mean_case",
                 symbol="comparison",
                 hover_name="genes")
fig.show()

## Size of the dots

The size of the points in a scatter plot can be set with the `size` argument.

Choose that variable carefully. It is tempting to write `size="p_value"`, but that makes the *least* significant proteins the biggest dots, which is the opposite of what a reader expects. Mapping size to `neg_log10_p` instead makes the strongest evidence the most visible.

In [ ]:
fig = px.scatter(top_per_comparison,
                 x="genes",
                 y="log2_fold_change",
                 size="neg_log10_p",
                 color="comparison",
                 labels={"log2_fold_change": "log2 fold change",
                         "neg_log10_p": "-log10(p)"},
                 title="Strongest changes per comparison (dot size = significance)")
fig.show()

## The volcano plot

The volcano plot is the most recognisable figure in omics, and it is nothing more than a scatter plot of the two numbers every differential test produces:

- **x = log2 fold change**: how big is the difference, and in which direction,
- **y = -log10(p-value)**: how strong is the evidence that the difference is real.

Effect size and evidence are different questions, and the volcano plot is the figure that refuses to confuse them. A protein can double in abundance on the strength of one noisy patient (far right, low down) or change by 10 % with beautiful consistency (near the centre, high up). Neither is a biomarker on its own.

Two conventions make the plot readable:
- a **horizontal line** at the significance threshold, and
- a **symmetric x axis with a vertical line at zero**, so "up" and "down" are visually comparable.

One honest caveat about our table: `published_deps.tsv` contains only the proteins the authors reported as significant, so the dense cloud of unchanged proteins that normally fills the bottom of a volcano is missing. Our plot shows the two wings of the volcano without its base. With a full result table you would see the classic shape.

In [ ]:
volcano = deps_pairwise[deps_pairwise["comparison"] == "CRKP_vs_Con"]

fig = px.scatter(volcano,
                 x="log2_fold_change",
                 y="neg_log10_p",
                 color="direction",
                 color_discrete_map={"up in cases": "#C44E52", "down in cases": "#4C72B0"},
                 hover_name="genes",
                 hover_data=["protein_group", "p_value"],
                 labels={"log2_fold_change": "log2 fold change (CRKP / Con)",
                         "neg_log10_p": "-log10(p-value)",
                         "direction": ""},
                 title="Volcano plot: resistant K. pneumoniae sepsis vs culture-negative sepsis")

# The significance threshold and the line of no change
fig.add_hline(y=-np.log10(0.05), line_dash="dash", line_color="grey",
              annotation_text="p = 0.05")
fig.add_vline(x=0, line_dash="dot", line_color="grey")

# A symmetric x axis, so that up and down are visually comparable
limit = volcano["log2_fold_change"].abs().max() * 1.1
fig.update_layout(xaxis_range=[-limit, limit], template="plotly_white")
fig.show()

### Labelling the interesting points

A volcano plot with 293 dots is useful only if the reader can find out which protein a dot is. There are two complementary ways to do that, and a good figure often uses both:

- **hovering** (interactive): we already set `hover_name="genes"` and `hover_data=[...]`, so every point tells you its gene name, its protein group accession and its exact p-value. Hover over the top-right corner of the plot above.
- **printed labels** (works on paper): only for the handful of points that carry the message. Labelling all 293 would produce an unreadable mess, which is exactly the "chartjunk" a figure should avoid.

Below we print the names of the 10 most significant proteins and leave the rest to the hover.

In [ ]:
top_hits = volcano.nlargest(10, "neg_log10_p")

fig = px.scatter(volcano,
                 x="log2_fold_change",
                 y="neg_log10_p",
                 color="direction",
                 color_discrete_map={"up in cases": "#C44E52", "down in cases": "#4C72B0"},
                 hover_name="genes",
                 hover_data=["protein_group", "p_value"],
                 opacity=0.6,
                 labels={"log2_fold_change": "log2 fold change (CRKP / Con)",
                         "neg_log10_p": "-log10(p-value)",
                         "direction": ""},
                 title="CRKP vs Con, 10 most significant proteins labelled")

for _, row in top_hits.iterrows():
    fig.add_annotation(x=row["log2_fold_change"], y=row["neg_log10_p"],
                       text=row["genes"], showarrow=True, arrowhead=0,
                       ax=15, ay=-15, font=dict(size=10))

fig.add_hline(y=-np.log10(0.05), line_dash="dash", line_color="grey")
fig.update_layout(template="plotly_white")
fig.show()

## Trend line

We can add a trend line with the `trendline` argument, passing it the model to use. The default is an ordinary least squares fit (linear regression), with the value `ols`.

A regression line only makes sense between two continuous variables, so we move from the protein table to the clinical table and ask a simple question: does the inflammatory marker CRP rise with the age of the patient?

(`trendline="ols"` needs the `statsmodels` package, which is already installed in Google Colab.)

In [ ]:
fig = px.scatter(meta,
                 x="age",
                 y="c_reactive_protein",
                 color="group",
                 color_discrete_map=GROUP_COLOURS,
                 category_orders={"group": GROUP_ORDER},
                 trendline="ols",
                 trendline_scope="overall",
                 trendline_color_override="black",
                 hover_name="sample_id",
                 labels={"age": "Age (years)",
                         "c_reactive_protein": "C-reactive protein (mg/L)"},
                 title="CRP against age, one point per patient")
fig.show()

## A quality control scatter: mean against variability

The same plot type answers a very different question if you change the axes. For each protein we compute, across the 45 patients:

- the **mean intensity** (how abundant is it), and
- the **coefficient of variation** (CV = standard deviation / mean, in per cent, how variable is it).

The expected shape is a decreasing cloud: abundant proteins are measured precisely, low-abundance ones are noisy. Points that sit high and to the right are worth a second look before you trust them as biomarkers.

Two design decisions do the work here. Raw intensities span five orders of magnitude, so on a linear axis 95 % of the proteins are squeezed against the left edge; `log_x=True` spreads them out. And the colour carries a third variable, the number of patients in which the protein was actually detected.

In [ ]:
intensities = prot[patient_cols]

prot_stats = pd.DataFrame({
    "protein_group": prot["protein_group"],
    "genes": prot["genes"],
    "mean_intensity": intensities.mean(axis=1),
    "cv_percent": 100 * intensities.std(axis=1) / intensities.mean(axis=1),
    "n_detected": intensities.notna().sum(axis=1),
})

fig = px.scatter(prot_stats,
                 x="mean_intensity",
                 y="cv_percent",
                 color="n_detected",
                 log_x=True,
                 opacity=0.7,
                 hover_name="genes",
                 color_continuous_scale="Viridis",
                 labels={"mean_intensity": "mean intensity (log scale)",
                         "cv_percent": "coefficient of variation (%)",
                         "n_detected": "patients with a value"},
                 title="Abundance against variability, 1458 protein groups")
fig.update_layout(template="plotly_white")
fig.show()

## Exercises

1) Find the option that removes the legend from this figure.

In [ ]:
fig = px.scatter(top_per_comparison,
                 x="genes",
                 y="log2_fold_change",
                 size="neg_log10_p",
                 color="comparison")
fig.show()

2) Modify the previous plot to use symbols that distinguish the direction of change (`direction`). If the figure becomes hard to read, filter the data down to a single comparison.

# Bar charts

The function `px.bar()` draws bar charts, which show a quantitative value for each level of a qualitative variable.

The most common bar chart in an omics paper is a **count**: how many patients per group, how many metabolites per chemical class, how many proteins quantified per sample. The counting is done in pandas; Plotly only draws the result.

We start with the composition of the cohort. `groupby(...).size()` gives the number of patients for each combination of group and sex.

In [ ]:
group_counts = (meta
                .groupby(["group", "sex"], as_index=False)
                .size()
                .rename(columns={"size": "n_patients"}))
group_counts

By default the bars of the different colours are stacked on top of each other, so the total height is the number of patients in the group.

In [ ]:
fig = px.bar(group_counts,
             x="group",
             y="n_patients",
             color="sex",
             category_orders={"group": GROUP_ORDER},
             labels={"n_patients": "patients", "group": ""},
             title="Cohort composition: 15 patients per group")
fig.show()

In [ ]:
fig = px.bar(group_counts,
             x="group",
             y="n_patients",
             color="sex",
             barmode="group",
             category_orders={"group": GROUP_ORDER},
             labels={"n_patients": "patients", "group": ""},
             title="Cohort composition, bars side by side")
fig.show()

## Orientation

The bar plot is vertical by default, but it can be drawn horizontally by setting the `orientation` argument to `"h"`.

This is not a matter of taste. Whenever the category names are long, horizontal bars let you write them out in full and read them left to right, instead of rotating them 90 degrees. The chemical classes of the metabolomics annotation are a perfect example.

In [ ]:
annotation = pd.read_csv(f"{BASE_URL}/metabolomics/data/metabolite_annotation.tsv", sep="\t")

class_counts = (annotation["class_i"]
                .value_counts()
                .head(15)
                .rename_axis("class_i")
                .reset_index(name="n_metabolites")
                .sort_values("n_metabolites"))

fig = px.bar(class_counts,
             x="n_metabolites",
             y="class_i",
             orientation="h",
             labels={"n_metabolites": "annotated metabolites", "class_i": ""},
             title="Chemical classes in the metabolomics annotation")
fig.show()

## Adding text to the bars

We can write a value on each bar with the `text` argument, passing it any variable of the DataFrame. `text_auto` controls the number format.

Printed values work when there are few bars. With 45 bars they turn into noise, so use them for summaries, not for per-sample plots.

In [ ]:
group_totals = (meta["group"]
                .value_counts()
                .rename_axis("group")
                .reset_index(name="n_patients"))

fig = px.bar(group_totals,
             x="group",
             y="n_patients",
             text="n_patients",
             color="group",
             color_discrete_map=GROUP_COLOURS,
             category_orders={"group": GROUP_ORDER},
             labels={"n_patients": "patients", "group": ""},
             title="Patients per group")
fig.update_layout(showlegend=False)
fig.show()

## A bar chart per sample

Counting the non-missing values of each column of the protein matrix gives the number of protein groups quantified in each patient. This is one of the first plots to look at in any proteomics experiment: a sample with far fewer identifications than its neighbours is a technical problem, not a biological finding.

Note two things: the samples are sorted by group (using `group_order` from the metadata) so the eye can compare groups, and the bars are coloured with the palette we defined at the top. The QC pools are excluded, because they are not patients.

In [ ]:
ids_per_sample = (prot[patient_cols]
                  .notna()
                  .sum()
                  .rename_axis("sample_id")
                  .reset_index(name="n_proteins")
                  .merge(meta[["sample_id", "group", "group_order"]], on="sample_id")
                  .sort_values(["group_order", "sample_id"]))

fig = px.bar(ids_per_sample,
             x="sample_id",
             y="n_proteins",
             color="group",
             color_discrete_map=GROUP_COLOURS,
             category_orders={"group": GROUP_ORDER},
             hover_data=["n_proteins"],
             labels={"sample_id": "", "n_proteins": "protein groups quantified"},
             title="Protein groups quantified per patient")
fig.update_layout(template="plotly_white")
fig.show()

# Histograms

Histograms show the distribution of a variable. In Plotly a histogram is close to a bar plot in which the data are aggregated with one of several possible functions (sum, average, count, ...).

Compared with `px.bar()`, `px.histogram()` can work with the `x` argument alone, which may be a continuous or a categorical variable. If you also give it `y` and a `histfunc`, it aggregates that variable within each bin: the first example below shows the **mean CRP of each group**, computed by Plotly itself.

In [ ]:
fig = px.histogram(meta,
                   x="group",
                   y="c_reactive_protein",
                   histfunc="avg",
                   color="group",
                   color_discrete_map=GROUP_COLOURS,
                   category_orders={"group": GROUP_ORDER},
                   labels={"group": "", "c_reactive_protein": "CRP (mg/L)"},
                   title="Mean C-reactive protein per group")
fig.update_layout(showlegend=False, yaxis_title="mean CRP (mg/L)")
fig.show()

## Why we plot log2 intensities

Now the distribution of the measurements themselves. To feed the matrix to Plotly we reshape it from **wide** (one column per sample) to **long** (one row per protein and sample) with `melt()`, and attach the group of each sample.

Then we plot the raw intensities. The result is the shape every mass spectrometry intensity distribution has: a spike against the left axis and a long tail reaching to the right edge of the plot, in which no structure whatsoever can be seen. This is why intensities are almost always log-transformed before plotting or testing.

In [ ]:
prot_long = (prot
             .melt(id_vars=["protein_group", "genes"],
                   value_vars=patient_cols,
                   var_name="sample_id",
                   value_name="intensity")
             .dropna(subset=["intensity"])
             .merge(meta[["sample_id", "group", "group_order"]], on="sample_id"))

prot_long["log2_intensity"] = np.log2(prot_long["intensity"])

print(prot_long.shape)
fig = px.histogram(prot_long,
                   x="intensity",
                   nbins=100,
                   labels={"intensity": "raw intensity"},
                   title="Raw protein intensities: unreadable on a linear scale")
fig.show()

In [ ]:
fig = px.histogram(prot_long,
                   x="log2_intensity",
                   nbins=50,
                   color="group",
                   color_discrete_map=GROUP_COLOURS,
                   category_orders={"group": GROUP_ORDER},
                   opacity=0.7,
                   labels={"log2_intensity": "log2 intensity", "group": ""},
                   title="The same data on a log2 scale")
fig.update_layout(barmode="overlay", xaxis_title="log2 intensity",
                  yaxis_title="protein measurements", template="plotly_white")
fig.show()

# Heatmaps

A heatmap shows a whole matrix at once, with colour standing in for the value. `px.imshow()` draws one from a DataFrame.

Three decisions turn a heatmap from a coloured rectangle into a figure:

1. **Choose the rows.** All 1458 proteins would give one pixel per row and no information. We keep the 30 proteins with the largest variance across patients, because those are the ones that can separate anything.
2. **Scale each row.** Proteins differ in abundance by orders of magnitude, so without scaling the map only shows which proteins are abundant. We z-score each row (subtract its mean, divide by its standard deviation), so the colour means "high or low **for this protein**".
3. **Choose the colour scale to match the data.** z-scores are symmetric around zero, so they need a *diverging* scale with a neutral middle (`RdBu_r`, centred on 0). A sequential scale here would suggest that zero is unremarkable in the same way that -3 is. We also fix the limits with `zmin`/`zmax` so a single extreme value cannot wash out the rest.

The samples are ordered by group, and dashed lines mark the group boundaries, which is how we annotate the columns without a second colour bar.

In [ ]:
# 1. log2 matrix of the proteins quantified in every patient
log2_matrix = np.log2(prot.set_index("protein_group")[patient_cols])
complete = log2_matrix.dropna()

# 2. the 30 most variable proteins, z-scored row by row
most_variable = complete.loc[complete.var(axis=1).nlargest(30).index]
zscores = most_variable.sub(most_variable.mean(axis=1), axis=0).div(
    most_variable.std(axis=1), axis=0)

# 3. samples ordered by group, rows labelled with the gene name
sample_order = meta.sort_values(["group_order", "sample_id"])["sample_id"].tolist()
zscores = zscores[sample_order]
gene_labels = prot.set_index("protein_group")["genes"].reindex(zscores.index).tolist()

fig = px.imshow(zscores.to_numpy(),
                x=sample_order,
                y=gene_labels,
                color_continuous_scale="RdBu_r",
                zmin=-3, zmax=3,
                aspect="auto",
                labels={"x": "", "y": "", "color": "z-score"},
                title="30 most variable proteins, samples grouped by condition")

# Dashed lines and labels marking the three groups
boundary = 0
for group in GROUP_ORDER:
    n = (meta["group"] == group).sum()
    fig.add_annotation(x=boundary + n / 2 - 0.5, y=1.02, xref="x", yref="paper",
                       text=group, showarrow=False, font=dict(size=13))
    boundary += n
    if boundary < len(sample_order):
        fig.add_vline(x=boundary - 0.5, line_dash="dash", line_color="black")

fig.update_layout(height=700)
fig.show()

# Box plots and violin plots

Box plots and violin plots are a good way of showing distributions. `px.box()` and `px.violin()` draw them and work in exactly the same way (same arguments).

The box shows the median and the quartiles, so it is a five-number summary; the violin shows the whole estimated density, so it also reveals a distribution with two peaks, which a box plot would hide. With only 15 patients per group it is good practice to show the individual points as well.

We start with procalcitonin, the clinical marker of bacterial infection, in the three groups.

In [ ]:
fig = px.box(meta,
             y="procalcitonin",
             x="group",
             category_orders={"group": GROUP_ORDER},
             labels={"group": "", "procalcitonin": "procalcitonin (ng/mL)"},
             title="Procalcitonin per group")
fig.show()

In [ ]:
# The same plot type as a quality control figure: one box per sample
fig = px.box(prot_long,
             x="sample_id",
             y="log2_intensity",
             color="group",
             color_discrete_map=GROUP_COLOURS,
             category_orders={"group": GROUP_ORDER,
                              "sample_id": ids_per_sample["sample_id"].tolist()},
             labels={"sample_id": "", "log2_intensity": "log2 intensity", "group": ""},
             title="Distribution of log2 intensities in each patient")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
fig = px.violin(meta,
                y="procalcitonin",
                x="group",
                category_orders={"group": GROUP_ORDER},
                labels={"group": "", "procalcitonin": "procalcitonin (ng/mL)"},
                title="Procalcitonin per group")
fig.show()

In [ ]:
fig = px.violin(meta,
                y="procalcitonin",
                x="group",
                color="group",
                color_discrete_map=GROUP_COLOURS,
                category_orders={"group": GROUP_ORDER},
                points="all",
                hover_name="sample_id",
                labels={"group": "", "procalcitonin": "procalcitonin (ng/mL)"},
                title="Procalcitonin per group, all patients shown")
fig.show()

## Box plot inside the violin

In [ ]:
fig = px.violin(meta,
                y="procalcitonin",
                x="group",
                color="group",
                color_discrete_map=GROUP_COLOURS,
                category_orders={"group": GROUP_ORDER},
                box=True,
                labels={"group": "", "procalcitonin": "procalcitonin (ng/mL)"},
                title="Procalcitonin per group, violin with box")
fig.show()

## Notched box plot

The notch is a rough confidence interval around the median: when the notches of two boxes do not overlap, the medians are probably different. It is a visual test, not a substitute for a statistical one.

In [ ]:
fig = px.box(meta,
             y="c_reactive_protein",
             x="group",
             color="group",
             color_discrete_map=GROUP_COLOURS,
             category_orders={"group": GROUP_ORDER},
             notched=True,
             points="all",
             hover_name="sample_id",
             labels={"group": "", "c_reactive_protein": "CRP (mg/L)"},
             title="C-reactive protein per group")
fig.show()

## Showing the mean

`boxmean=True` adds the mean as a dashed line inside the box. Comparing it with the median is a quick check for skew: if the mean sits well above the median, a few extreme patients are pulling it up.

In [ ]:
fig = px.box(meta,
             y="c_reactive_protein",
             x="group",
             color="group",
             color_discrete_map=GROUP_COLOURS,
             category_orders={"group": GROUP_ORDER},
             notched=True,
             points="all",
             labels={"group": "", "c_reactive_protein": "CRP (mg/L)"},
             title="C-reactive protein per group, mean shown as a dashed line")
fig.update_traces(boxmean=True)
fig.show()

## Log scales for skewed variables

Procalcitonin in this cohort runs from 0.02 to 89 ng/mL: a factor of more than 4000. On a linear axis the three or four highest patients own the whole plot and every value below 5 ng/mL is flattened onto the baseline, so the difference between the groups is invisible.

Setting `log_y=True` gives each order of magnitude the same amount of space. Nothing about the data changed, only the ruler we measure it with, and the group difference becomes obvious. Always say in the axis label or the caption that the scale is logarithmic.

In [ ]:
fig = px.box(meta,
             y="procalcitonin",
             x="group",
             color="group",
             color_discrete_map=GROUP_COLOURS,
             category_orders={"group": GROUP_ORDER},
             points="all",
             log_y=True,
             hover_name="sample_id",
             labels={"group": "", "procalcitonin": "procalcitonin (ng/mL, log scale)"},
             title="Procalcitonin per group on a logarithmic scale")
fig.update_layout(showlegend=False, template="plotly_white")
fig.show()

## Small multiples (faceting)

When you want to show the same plot for several proteins, do not squeeze them into one crowded axis: repeat the plot once per protein with `facet_col`. The panels share axes and colours, so the reader learns the layout once and then compares the panels.

Here we take four acute-phase serum proteins and show their log2 intensity in the three groups. `facet_col_wrap` starts a new row after a given number of panels.

In [ ]:
genes_of_interest = ["SAA1", "SAA2", "LBP", "CRP"]
subset = prot_long[prot_long["genes"].isin(genes_of_interest)]

fig = px.box(subset,
             x="group",
             y="log2_intensity",
             color="group",
             color_discrete_map=GROUP_COLOURS,
             category_orders={"group": GROUP_ORDER, "genes": genes_of_interest},
             facet_col="genes",
             facet_col_wrap=2,
             points="all",
             hover_name="sample_id",
             labels={"group": "", "log2_intensity": "log2 intensity"},
             title="Acute-phase proteins across the three groups")
fig.update_layout(showlegend=False, template="plotly_white", height=650)
fig.show()

## Error bars

In **scatter**, **line** and **bar** plots it is important to show the uncertainty of the value, for example a confidence interval or an error metric. In Plotly you can use the `error` argument (`error_x` or `error_y` depending on the axis) and the error bar is added to the figure.

**Note**: to show an error you need a second variable holding the size of the error. Here we compute a real one: the standard error of the mean (SEM = standard deviation / square root of the number of patients) of CRP in each group.

A bar chart of group means without error bars is the classic misleading omics figure: three bars of different heights always look like a finding. The error bars are what let the reader judge whether 15 patients per group support that impression. Say in the caption what the bars are, since a standard deviation, a standard error and a 95 % confidence interval look identical and mean different things.

In [ ]:
group_stats = (meta
               .groupby("group")["c_reactive_protein"]
               .agg(mean_crp="mean", sd_crp="std", n="size")
               .reset_index())
group_stats["sem_crp"] = group_stats["sd_crp"] / np.sqrt(group_stats["n"])

fig = px.bar(group_stats,
             x="group",
             y="mean_crp",
             color="group",
             color_discrete_map=GROUP_COLOURS,
             category_orders={"group": GROUP_ORDER},
             error_y="sem_crp",
             labels={"group": "", "mean_crp": "mean CRP (mg/L)"},
             title="Mean C-reactive protein per group (error bars: SEM, n = 15)")
fig.update_layout(showlegend=False, template="plotly_white")
fig.show()

group_stats

## Animating your figures

Plotly Express can build animations with the `animation_frame` and `animation_group` arguments. Have a look at the examples here: https://plotly.com/python/animations/.

An animation needs a variable to step through. Time is the usual one; here we step through the three **comparisons**, so the same set of genes is redrawn as we move from "susceptible infection versus culture-negative" to "resistant versus culture-negative" to "resistant versus susceptible". Genes that were not significant in a given comparison are shown as zero, which is what `fillna(0)` does below, and what the caption must say.

Note the colour: the original of this example coloured each bar by the category already given by the y axis, which adds nothing. Colouring by the **direction** of the change instead adds real information, and two colours are easier to read than twelve.

In [ ]:
animation_genes = top20_crkp_con["genes"].tail(12).tolist()

animation_data = (deps_pairwise[deps_pairwise["genes"].isin(animation_genes)]
                  .pivot_table(index="genes", columns="comparison",
                               values="log2_fold_change")
                  .reindex(index=animation_genes,
                           columns=["KP_vs_Con", "CRKP_vs_Con", "CRKP_vs_KP"])
                  .fillna(0)
                  .stack()
                  .rename("log2_fold_change")
                  .reset_index())
animation_data["direction"] = np.where(animation_data["log2_fold_change"] >= 0,
                                       "up in cases", "down in cases")

fig = px.bar(animation_data,
             y="genes",
             x="log2_fold_change",
             color="direction",
             color_discrete_map={"up in cases": "#C44E52", "down in cases": "#4C72B0"},
             orientation="h",
             animation_frame="comparison",
             animation_group="genes",
             range_x=[-5, 5],
             labels={"genes": "", "log2_fold_change": "log2 fold change",
                     "direction": ""},
             title="Fold change of 12 proteins across the three comparisons")
fig.update_layout(template="plotly_white", height=550)
fig.show()

## Saving your plots

To save plots in a static format (for example `png`) you first have to install the [Kaleido](https://pypi.org/project/kaleido/) library. Once it is installed you can use the `write_image()` method, giving it the file name (path).

Saving to `html` with `write_html()` needs no extra library and keeps the interactivity, so the reader can still hover and read gene names. Use `png` (or `pdf`) for the manuscript figure, and `html` for the version you send to your collaborators.

**Note:** in Colab you have to restart the session for the Kaleido installation to take effect (Runtime -> Restart session).

In [ ]:
fig.write_image("log2fc_per_comparison.png")   # static, needs kaleido
fig.write_html("log2fc_per_comparison.html")   # interactive, no extra library

## Exercise

1) Build a volcano plot for the comparison `CRKP_vs_KP` (resistant versus susceptible infection), the comparison that answers the clinical question of the study. Label the axes, add the significance threshold, and make the hover show the gene name and the p-value.

2) Turn it into an animation over the three comparisons with `animation_frame="comparison"`. Choose the axis ranges yourself, and remember that they must be the same in every frame for the animation to be comparable.

# Other resources

1. [Matplotlib](https://matplotlib.org/)
2. [Bokeh](https://bokeh.org/)
3. [Altair](https://altair-viz.github.io/)
4. [Seaborn](https://seaborn.pydata.org/)
5. [Python Graph Gallery](https://www.python-graph-gallery.com/)
6. [Plotly Express documentation](https://plotly.com/python/plotly-express/)